# LLM Engineering: Company Brochure Generator

## Project Overview

This notebook demonstrates **advanced LLM engineering techniques** by building an automated system that:

1. **Scrapes company websites** and extracts structured content
2. **Uses intelligent link filtering** via LLM (one-shot prompting) to identify relevant pages
3. **Generates polished brochures** by combining multiple API calls
4. **Streams results** for real-time output with typewriter animation

## LLM Skills Demonstrated

| Skill | Implementation |
|-------|-----------------|
| **Web Scraping + LLM Analysis** | Website class + GPT-4o-mini to intelligently filter links |
| **Prompt Engineering** | One-shot prompting with JSON structured outputs |
| **Chaining API Calls** | Multi-step pipeline: fetch → analyze → generate → format |
| **Streaming Output** | Real-time response streaming with dynamic display updates |
| **Error Handling & Validation** | API key verification, content extraction fallbacks |
| **Production Patterns** | Reusable classes, environment variable management, modular design |

## Quick Demo

Run all cells in sequence. The notebook will:
- Scrape a real company website (e.g., Anthropic, HuggingFace)
- Extract relevant pages automatically
- Generate a markdown brochure in real-time

**Example output:** Professional company brochure with culture, customers, and career info.

---

# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [3]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)                                          #load env var from .env file
api_key = os.getenv('OPENAI_API_KEY')                               #fetch the value within 'OPENAI_API_KEY'

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:  #api key always starts with 'sk-proj-' and length checking to make sure it's not too short
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


#### about user agent
This is a User-Agent header that we attach when making HTTP requests. It tells the server what kind of client we are — for example, a web browser like Chrome. Many websites block or give limited content to requests that don’t look like real browsers. By setting a proper User-Agent string, our request looks like it’s coming from a normal browser, which helps avoid being blocked and ensures we get the full webpage content.

#### why it looks so weird (Mozilla, AppleWebKit, Chrome, Safari all together):

The User-Agent string is a legacy mix. It includes terms like Mozilla for historical compatibility, AppleWebKit because Chrome is based on the WebKit engine, and Safari/Chrome to show compatibility with both. Websites often check for different pieces of this string, so browsers include all of them to avoid being rejected.

In [5]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)           #use request package to retrive url
        self.body = response.content                           #collecte content
        soup = BeautifulSoup(self.body, 'html.parser')         #parsing with Beautiful soup
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]): #getting content , tittle and stripping out junk we dont need
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]   #gather all the links that reffers on tis page
        self.links = [link for link in links if link]                #collect them in self.links

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"


In [6]:
huggingface_site = Website("https://huggingface.co")
# print(huggingface_site.get_contents())
# print(huggingface_site.links)

huggingface_site.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/Qwen/Qwen3.6-35B-A3B',
 '/tencent/HY-Embodied-0.5',
 '/MiniMaxAI/MiniMax-M2.7',
 '/unsloth/Qwen3.6-35B-A3B-GGUF',
 '/baidu/ERNIE-Image',
 '/models',
 '/spaces/k2-fsa/OmniVoice',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview',
 '/spaces/webml-community/bonsai-webgpu',
 '/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast',
 '/spaces/openbmb/VoxCPM-Demo',
 '/spaces',
 '/datasets/lambda/hermes-agent-reasoning-traces',
 '/datasets/Roman1111111/claude-opus-4.6-10000x',
 '/datasets/llamaindex/ParseBench',
 '/datasets/ianncity/KIMI-K2.5-1000000x',
 '/datasets/Kassadin88/GLM-5.1-1000000x',
 '/datasets',
 '/join',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/inference/models',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel'

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [7]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""



#these types of commands which are provided witth one examples,
# are called one shot promting

In [8]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [9]:
# def get_links_user_prompt(website):
#     user_prompt = f"Here is the list of links on the website of {website.url} - "
#     user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
# Do not include Terms of Service, Privacy, email links.\n"
#     user_prompt += "Links (some might be relative links):\n"
#     user_prompt += "\n".join(website.links)
#     return user_prompt


def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [10]:
print(get_links_user_prompt(huggingface_site))

Here is the list of links on the website of https://huggingface.co - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/login
/join
/spaces
/models
/Qwen/Qwen3.6-35B-A3B
/tencent/HY-Embodied-0.5
/MiniMaxAI/MiniMax-M2.7
/unsloth/Qwen3.6-35B-A3B-GGUF
/baidu/ERNIE-Image
/models
/spaces/k2-fsa/OmniVoice
/spaces/r3gm/wan2-2-fp8da-aoti-preview
/spaces/webml-community/bonsai-webgpu
/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast
/spaces/openbmb/VoxCPM-Demo
/spaces
/datasets/lambda/hermes-agent-reasoning-traces
/datasets/Roman1111111/claude-opus-4.6-10000x
/datasets/llamaindex/ParseBench
/datasets/ianncity/KIMI-K2.5-1000000x
/datasets/Kassadin88/GLM-5.1-1000000x
/datasets
/join
/enterprise
/enterprise
/enterprise
/enterprise
/enterprise
/enterprise
/enter

In [11]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [12]:


huggingface_site = Website("https://huggingface.co")
huggingface_site.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/Qwen/Qwen3.6-35B-A3B',
 '/tencent/HY-Embodied-0.5',
 '/MiniMaxAI/MiniMax-M2.7',
 '/unsloth/Qwen3.6-35B-A3B-GGUF',
 '/baidu/ERNIE-Image',
 '/models',
 '/spaces/k2-fsa/OmniVoice',
 '/spaces/r3gm/wan2-2-fp8da-aoti-preview',
 '/spaces/webml-community/bonsai-webgpu',
 '/spaces/prithivMLmods/FireRed-Image-Edit-1.0-Fast',
 '/spaces/openbmb/VoxCPM-Demo',
 '/spaces',
 '/datasets/lambda/hermes-agent-reasoning-traces',
 '/datasets/Roman1111111/claude-opus-4.6-10000x',
 '/datasets/llamaindex/ParseBench',
 '/datasets/ianncity/KIMI-K2.5-1000000x',
 '/datasets/Kassadin88/GLM-5.1-1000000x',
 '/datasets',
 '/join',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/inference/models',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel'

In [ ]:
# get_links("https://huggingface.co")

get_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://www.anthropic.com/company'},
  {'type': 'careers page', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'research page', 'url': 'https://www.anthropic.com/research'},
  {'type': 'economic futures page',
   'url': 'https://www.anthropic.com/economic-futures'},
  {'type': 'transparency page',
   'url': 'https://www.anthropic.com/transparency'},
  {'type': 'events page', 'url': 'https://www.anthropic.com/events'}]}

 ## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [13]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [14]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}
Landing page:
Webpage Title:
Hugging Face – The AI community building the future.
Webpage Contents:
Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M

In [15]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [16]:
print(MODEL)

gpt-4o-mini


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:80_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog', 'url': 'https://huggingface.co/blog'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'status page', 'url': 'https://status.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.6-35B-A3B\nUpdated\n4 days ago\n•\n209k\n•\n873\ntencent/HY-Embodied-0.5\nUpdated\n5 days ago\n•\n1.6k\n•\n867\nMiniMaxAI/MiniMax-M2.7\nUpdated\n2 days ago\n•\n289k\n•\n967\nunsloth/Qwen3.6-35B-A3B-GGUF\nUpdated\n3 days ago\n•\n662k\n•\n475\nbaidu/ERNIE-Image\nUpdated\n2 days ago\n•\n3.76k\n•\n458\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nAgents\nFeatured\n599\nOmniVoice

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        temperature=0,      # Set to 0 for deterministic/consistent output (no randomness)
        seed=42             # Fixed seed ensures same results across runs
    )

    result = response.choices[0].message.content    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'company blog', 'url': 'https://huggingface.co/blog'}, {'type': 'community discussion', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


# Hugging Face Brochure

## 🌐 About Us
**Hugging Face** is an innovative AI community dedicated to building the future through research and collaboration in machine learning. Founded in 2016, Hugging Face has quickly become a core player in the AI ecosystem, focusing on making powerful technologies in natural language processing and machine learning accessible to everyone.

## ✨ Our Offerings
- **Models**: Explore a vast repository of over 2 million AI models, including state-of-the-art models for a range of applications—from text processing to image generation.
- **Datasets**: Access more than 500,000 datasets designed for various machine learning tasks.
- **Spaces**: Host and interact with AI applications, helping users to create, discover, and collaborate on ML projects seamlessly.
- **Enterprise Solutions**: With advanced security, dedicated support, and scalable access controls, our enterprise offerings cater to over 50,000 organizations worldwide, including major industry players like Google, Amazon, and Microsoft.

## ✔️ Careers at Hugging Face
Join a passionate team of about 185 professionals based in **NYC** and **Paris**. We are on the lookout for talented individuals committed to democratizing machine learning. Whether you're a developer, researcher, or creative, there’s a place for you in our rapidly growing organization. Check our [careers page](https://huggingface.co/careers) for current openings.

### **Company Culture**
At Hugging Face, we value collaboration and an open-source mindset. Our culture thrives on inclusivity, innovation, and a shared mission to enhance machine learning technologies for a global community. We strive to create an environment where every team member can contribute and make a difference.

## 🤝 Our Community
We are more than just a tech company; we are a thriving community of researchers, educators, and engineers. With a vibrant forum and community blogs, we foster knowledge sharing and continuous learning among community members. Join us as we collaboratively build the future of AI.

## 🌎 Our Clients
Hugging Face serves over **50,000 organizations** globally, including tech giants like:
- **Meta**
- **Amazon**
- **Google**
- **Microsoft**

## 📈 Why Choose Hugging Face?
1. **Open Source**: Committed to open-source development, we provide communities with tools that are accessible and easy to use.
2. **Comprehensive Solutions**: Our robust platform supports all aspects of the machine learning lifecycle, empowering users to focus on building and innovating.
3. **Community-Driven**: Collaborate with a diverse group of developers, researchers, and enthusiasts who are passionate about AI.

## 🔗 Connect with Us
Learn more about Hugging Face and stay updated with our latest developments:
- [Website](https://huggingface.co)
- [Blog](https://huggingface.co/blog)

Join us in our mission of "Democratizing AI, one commit at a time!" 🙌

--- 

*This brochure is designed for prospective customers, investors, and recruits interested in learning more about Hugging Face and its offerings.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True,
        temperature=0,      # Set to 0 for deterministic/consistent output (no randomness)
        seed=42             # Fixed seed ensures same results across runs
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [28]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'company social media', 'url': 'https://twitter.com/huggingface'}, {'type': 'company social media', 'url': 'https://www.linkedin.com/company/huggingface/'}, {'type': 'community forum', 'url': 'https://discuss.huggingface.co'}]}


# Hugging Face Brochure

## About Us
**Hugging Face** is a leading AI community dedicated to building the future of machine learning. Founded in 2016, we provide a collaborative platform where developers, researchers, and organizations can explore, create, and share machine learning models, datasets, and applications. Our mission is to democratize artificial intelligence and make it accessible to everyone.

## Our Offerings
- **Models**: Access over 2 million models, including state-of-the-art AI models for various applications.
- **Datasets**: Explore and share more than 500,000 datasets tailored for machine learning tasks.
- **Spaces**: Create and host interactive applications with our user-friendly interface.
- **Enterprise Solutions**: Tailored plans for organizations, offering advanced security, dedicated support, and scalable compute options.

## Company Culture
At Hugging Face, we foster a culture of collaboration, innovation, and inclusivity. Our team is passionate about open-source contributions and believes in the power of community-driven development. We encourage our employees to share their knowledge and grow together, making it a vibrant place for creativity and learning.

## Customers
We proudly serve over 50,000 organizations, including industry giants like Google, Microsoft, Amazon, and Meta. Our platform is trusted by researchers, developers, and enterprises alike, making it a cornerstone of the AI community.

## Careers
Join us in our mission to democratize machine learning! We are always looking for talented individuals who are passionate about AI and want to make a difference. Explore our current job openings and become part of a dynamic team that is shaping the future of technology.

### Current Openings
- Machine Learning Engineers
- Data Scientists
- Software Developers
- Research Scientists

## Join the Community
Become a part of the Hugging Face community today! Whether you are a developer, researcher, or simply an AI enthusiast, there’s a place for you here. Collaborate, learn, and contribute to the future of machine learning.

### Connect with Us
- [Website](https://huggingface.co)
- [LinkedIn](https://www.linkedin.com/company/huggingface)
- [Twitter](https://twitter.com/huggingface)
- [Discord](https://discord.gg/huggingface)

---

**Hugging Face**: The AI community building the future. Join us in making machine learning accessible to all!